# 01 – Daten: Entities & Zeitreihen

Dumps all data points from InfluxDB (`data/datapoints.xlsx`), loads the relevant signals
(`rc_house.entities.SIGNALS`), resamples them to 15 min as a zero-order hold (time-weighted mean,
0/1 signals become a duty cycle) and stores them in `data/house_15min.csv` for the RC identification.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from rc_house import influx
from rc_house.entities import SIGNALS, ROOMS, THERMOSTAT_230V

DATA = Path("../data"); DATA.mkdir(exist_ok=True)
START = "2026-09-21T08:00:00Z"   # entity rename finished, current names valid from here
EVERY = "15min"
pd.set_option("display.max_rows", 500, "display.width", 200)

## Alle Datenpunkte

In [ ]:
dp = influx.list_datapoints("-30d")
en = influx.entities(dp)
with pd.ExcelWriter(DATA / "datapoints.xlsx") as w:
    en.to_excel(w, sheet_name="entities", index=False)
    dp.to_excel(w, sheet_name="series", index=False)
print(len(dp), "series,", len(en), "entities")
en.domain.value_counts()

## Relevante Signale laden & resamplen

In [ ]:
raw = influx.load_raw(SIGNALS, START)
df = influx.resample_hold(raw, EVERY)
df.to_csv(DATA / "house_15min.csv")
pd.DataFrame({"samples": {k: len(v) for k, v in raw.items()},
              "first": {k: v.index[0] for k, v in raw.items()},
              "last": {k: v.index[-1] for k, v in raw.items()}})

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
for r in ROOMS:
    ax[0].plot(df.index, df[f"T_{r}"], lw=0.8, label=r)
if "T_aussen" in df:
    ax[0].plot(df.index, df["T_aussen"], "k", lw=1, label="außen")
ax[0].set_ylabel("Temperatur [°C]"); ax[0].legend(ncol=7, fontsize=8)
for i, r in enumerate(ROOMS):
    ax[1].fill_between(df.index, i, i + 0.9 * df[f"heat_{r}"].fillna(0), step="post", lw=0)
ax[1].set_yticks([i + 0.45 for i in range(len(ROOMS))], list(ROOMS)); ax[1].set_title("Heizanforderung (hvac_action = heating)")
for r in sorted(THERMOSTAT_230V):
    ax[2].plot(df.index, df[f"pwr_{r}"], lw=0.8, label=r)
ax[2].set_ylabel("Versorgung 230V [-]"); ax[2].legend(fontsize=8)
plt.tight_layout()

## Raumthermostat II 230V – Versorgung

Wohnzimmer and Gastzimmer report 0.0 °C while the heating control cuts their supply.
When does that happen? (mean over time of day, local time)

In [ ]:
p = df[[f"pwr_{r}" for r in sorted(THERMOSTAT_230V)]].copy()
p.index = p.index.tz_convert("Europe/Vienna")
p.groupby(p.index.hour).mean().plot(kind="bar", figsize=(12, 3), title="Anteil versorgt je Stunde")
print("corr:", p.corr().iloc[0, 1].round(3))

## Heizraum – Vor-/Rücklauf (Shelly i4 Gen3, gemeinsamer Rücklauf)

In [ ]:
hz = df[["T_VL_og", "T_VL_eg", "T_RL"]].dropna(how="all")
ax = hz.plot(figsize=(12, 3), title="Vor-/Rücklauf")
ax.set_ylabel("°C"); df.filter(like="shelly_in").dropna(how="all").describe()